In [1]:
import os
import subprocess

command = "source ~/.zprofile && env"
proc = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    shell=True,
    executable="/bin/zsh"
)

for line in proc.stdout:
    key, _, value = line.decode().partition("=")
    os.environ[key] = value.strip()

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "aws_documentation": {
        "transport": "stdio",  # Key change: use stdio, not http
        "command": "uvx",
        "args": ["awslabs.aws-documentation-mcp-server@latest"],
        "env": {  # Optional environment variables
            "FASTMCP_LOG_LEVEL": "ERROR",
            "AWS_DOCUMENTATION_PARTITION": "aws"
        }
    }
})

# Later in your async code
tools = await client.get_tools()

In [3]:
tools

[StructuredTool(name='read_documentation', description="Fetch and convert an AWS documentation page to markdown format.\n\n## Usage\n\nThis tool retrieves the content of an AWS documentation page and converts it to markdown format.\nFor long documents, you can make multiple calls with different start_index values to retrieve\nthe entire content in chunks.\n\n## URL Requirements\n\n- Must be from the docs.aws.amazon.com domain\n- Must end with .html\n\n## Example URLs\n\n- https://docs.aws.amazon.com/AmazonS3/latest/userguide/bucketnamingrules.html\n- https://docs.aws.amazon.com/lambda/latest/dg/lambda-invocation.html\n\n## Output Format\n\nThe output is formatted as markdown text with:\n- Preserved headings and structure\n- Code blocks for examples\n- Lists and tables converted to markdown format\n\n## Handling Long Documents\n\nIf the response indicates the document was truncated, you have several options:\n\n1. **Continue Reading**: Make another call with start_index set to the end o

In [4]:
from langchain.agents import create_agent
agent = create_agent("openai:gpt-4.1", tools)
result = await agent.ainvoke({"messages": [{"role": "user", "content": "Tell me more about AWS S3 Services "}]})
print(f"Result: = {result}")

Result: = {'messages': [HumanMessage(content='Tell me more about AWS S3 Services ', additional_kwargs={}, response_metadata={}, id='75f4ea38-deb6-479a-a21e-1854f5ead130'), AIMessage(content='Amazon Simple Storage Service (Amazon S3) is a leading cloud object storage solution designed by Amazon Web Services (AWS). Here’s a concise overview of its core concepts, features, and use cases:\n\n### What is Amazon S3?\nAmazon S3 is an object storage service that offers industry-leading scalability, data availability, security, and performance. This means you can store and retrieve any amount of data, at any time, from anywhere on the web.\n\n### Key Features:\n- **Storage for the Internet:** Provides object storage through a web service interface.\n- **Durability and Reliability:** Designed for 99.999999999% (11 9’s) durability and 99.99% availability.\n- **Scalability:** Seamlessly scales storage resources up or down as needed.\n- **Security:** Supports encryption (in-transit and at-rest), AW

In [ ]:
print("stream mode test:")
async for event in agent.astream({
    "messages": [{"role": "user", "content": "Tell me more about MCP support in Agno"}]
}):
    print(event)